# kl-divergence-gaussian-closed-form — ex2: scalar 1-d KL closed form vs per-dim sum + batch mean — same number

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kl-divergence-gaussian-closed-form`. Running the final beacon cell reports progress against the `VAE: KL divergence Gaussian closed-form` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: KL divergence Gaussian closed-form` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kl-divergence-gaussian-closed-form`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kl-divergence-gaussian-closed-form"
DD_SUBTOPIC = "VAE: KL divergence Gaussian closed-form"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Gaussian KL — scalar 1-d closed form vs per-dim sum + batch mean

Ex1 computed the per-sample KL for a multi-dim diagonal Gaussian. The
deepening move shows the SAME formula in two equivalent forms:

**Form A — scalar 1-d closed form, for `q = N(mu, sigma^2)` against `N(0,1)`:**
```
kl_scalar = 0.5 * (mu^2 + sigma^2 - 1 - 2 * log(sigma))
          = 0.5 * (mu^2 + exp(2*logsigma) - 1 - 2 * logsigma)
```

**Form B — per-dim sum over a diagonal Gaussian, then mean over batch:**
```
kl_per_sample = -0.5 * sum_d(1 + 2*logsigma_d - mu_d^2 - exp(2*logsigma_d))
kl_batch_mean = kl_per_sample.mean()
```

**They agree numerically.** Form B applied to a single dim (sum over
one element) gives the same number as Form A for that dim. Run the
scalar form on each dim independently, sum them — must equal Form B's
per-sample KL.

**Sign sanity.** At `mu=0, logsigma=0` (which means sigma=1), the KL
is exactly 0 — `q == N(0, 1)` so no divergence. Any other (mu, logsigma)
gives a strictly positive value. Useful unit test.

### Exercise 2 — scalar 1-d KL closed form vs per-dim sum + batch mean — same number

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the equivalence between the scalar 1-D Gaussian KL formula applied element-wise and the per-dim-sum-then-batch-mean form by computing both on the same `(mu, logsigma)` tensor and verifying they agree numerically to machine precision.
> Keywords: kl-divergence, gaussian, closed-form, equivalence
> ```

**KCs targeted:** `scalar-kl-closed-form-from-mu-logsigma`, `per-dim-sum-equals-elementwise-sum`

Implement `ex2_kl_two_forms(mu, logsigma)` where `mu: (B, D)` and `logsigma: (B, D)`.

Compute BOTH forms of the diagonal-Gaussian KL vs `N(0, I)`:

**Form A — element-wise scalar closed form, shape `(B, D)`:**
```
kl_elem[b, d] = 0.5 * (mu[b, d]**2 + (2 * logsigma[b, d]).exp() - 1 - 2 * logsigma[b, d])
```

**Form B — per-sample KL (sum over dims), shape `(B,)`:**
```
kl_per_sample[b] = -0.5 * sum_d(1 + 2*logsigma[b,d] - mu[b,d]**2 - exp(2*logsigma[b,d]))
```

Then compute:
- `kl_batch_mean = kl_per_sample.mean()` — scalar.
- `kl_from_elem  = kl_elem.sum(dim=1)`  — `(B,)`, should equal Form B per sample.

Return:
```
{
    'kl_elem':         kl_elem,         # (B, D)
    'kl_per_sample':   kl_per_sample,   # (B,)
    'kl_from_elem':    kl_from_elem,    # (B,)
    'kl_batch_mean':   kl_batch_mean,   # scalar tensor
}
```


In [ ]:
def ex2_kl_two_forms(mu, logsigma):
    # Form A: element-wise scalar closed form, shape (B, D).
    kl_elem = 0.5 * (mu ** 2 + (2 * logsigma).exp() - 1 - 2 * logsigma)
    # Form B: per-sample sum, shape (B,).
    kl_per_sample = -0.5 * (1 + 2 * logsigma - mu ** 2 - (2 * logsigma).exp()).sum(dim=1)
    kl_from_elem  = kl_elem.sum(dim=1)
    kl_batch_mean = kl_per_sample.mean()
    return {
        'kl_elem':       kl_elem,
        'kl_per_sample': kl_per_sample,
        'kl_from_elem':  kl_from_elem,
        'kl_batch_mean': kl_batch_mean,
    }


<details><summary>Solution</summary>

```python
def ex2_kl_two_forms(mu, logsigma):
    # Form A: element-wise scalar closed form, shape (B, D).
    kl_elem = 0.5 * (mu ** 2 + (2 * logsigma).exp() - 1 - 2 * logsigma)
    # Form B: per-sample sum, shape (B,).
    kl_per_sample = -0.5 * (1 + 2 * logsigma - mu ** 2 - (2 * logsigma).exp()).sum(dim=1)
    kl_from_elem  = kl_elem.sum(dim=1)
    kl_batch_mean = kl_per_sample.mean()
    return {
        'kl_elem':       kl_elem,
        'kl_per_sample': kl_per_sample,
        'kl_from_elem':  kl_from_elem,
        'kl_batch_mean': kl_batch_mean,
    }
```

**The two forms are algebraically identical.** Distribute the negative half through Form B's parenthesis:
`-0.5 * (1 + 2*logsigma - mu^2 - exp(2*logsigma))`
`= -0.5 - logsigma + 0.5*mu^2 + 0.5*exp(2*logsigma)`
`= 0.5*(mu^2 + exp(2*logsigma) - 1 - 2*logsigma)`,
which is exactly Form A. Summing Form A over `dim=1` recovers Form B's per-sample value.

**`(2 * logsigma).exp()` over `logsigma.exp() ** 2`.** Same result, but the multiplied-then-exp form is numerically more stable for large `|logsigma|` and one fewer floating-point op.

**Why both forms are useful in practice.** `kl_elem` (the (B, D) tensor) shows you WHICH latent dims are doing the regularization work — collapsed dims have near-zero KL. `kl_per_sample` is the per-example value that goes into the loss. Both are computed cheaply from the same primitives.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()